In [14]:
import numpy as np 

In [15]:
class RNN:
    def __init__(self,input,hidden,output,lr):
        self.input_size = input
        self.hidden_size = hidden 
        self.output_size = output 
        self.lr = lr 
        self.Whh = np.random.randn(hidden,hidden)  
        self.Wxh = np.random.randn(hidden,input)
        self.Why = np.random.randn(output,hidden)
        self.Wbh = np.zeros(hidden)
        self.Wby = np.zeros(output)
    def softmax(self,z):

        z = z - np.max(z)
        z = np.exp(z)
        return z/np.sum(z)
    
    def forward(self, X):

        X = np.asarray(X, dtype=float)
        self.X = X
        self.T = len(X)
        self.h = {-1: np.zeros(self.hidden_size)}

        for t in range(self.T):
            z = (
                self.Wxh @ X[t]
                + self.Whh @ self.h[t - 1]
                + self.Wbh
            )
            self.h[t] = np.tanh(z)
        self.logits = self.Why @ self.h[self.T - 1] + self.Wby
        self.probs = self.softmax(self.logits)
        return self.probs
    
    def backward(self, y):

        # Softmax + cross-entropy gradient
        dy = self.probs.copy()
        dy[int(y)] -= 1.0

        # Output layer gradients
        dWhy = np.outer(dy, self.h[self.T - 1])
        dWby = dy.copy()

        # Gradient with respect to final hidden state
        dh = self.Why.T @ dy

        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWbh = np.zeros_like(self.Wbh)

        # Backpropagation through time
        for t in reversed(range(self.T)):
            da = dh * (1 - self.h[t] ** 2)

            dWxh += np.outer(da, self.X[t])
            dWhh += np.outer(da, self.h[t - 1])
            dWbh += da
            # Gradient to previous hidden state
            dh = self.Whh.T @ da
        # Gradient descent
        self.Wxh -= self.lr * dWxh
        self.Whh -= self.lr * dWhh
        self.Why -= self.lr * dWhy

        self.Wbh -= self.lr * dWbh
        self.Wby -= self.lr * dWby

    def loss(self,ypred,y):

        return -np.log(ypred[int(y)] + 1e-12)

    def train(self,epochs,X,y):

        for epoch in range(epochs):
            total_loss = 0 
            for sequence,target in zip(X,y):
                probs = self.forward(sequence)
                total_loss += self.loss(probs,target)
                self.backward(target)

            avg_loss = total_loss/len(X)
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch + 1}, "f"Loss: {avg_loss:.4f}")

    def test(self, X):
        probs = self.forward(X)
        prediction = np.argmax(probs)
        return prediction, probs

In [16]:
np.random.seed(42)

# 3 input features, 4 hidden units, 2 output classes
model = RNN(input=3, hidden=4, output=2, lr=0.01)

# Two training sequences, each with 3 time steps and 3 features
X_train = np.array([
    [[1, 0, 1], [0, 1, 0], [1, 1, 0]],
    [[0, 0, 1], [1, 0, 0], [0, 1, 1]]
], dtype=float)
print(X_train.shape)
y_train = np.array([0, 1])

model.train(epochs=100,X= X_train,y=y_train)

prediction, probabilities = model.test(X_train[0])

print("Probabilities:", probabilities)
print("Predicted class:", prediction)

(2, 3, 3)
Epoch 10, Loss: 1.0146
Epoch 20, Loss: 0.2784
Epoch 30, Loss: 0.1405
Epoch 40, Loss: 0.1069
Epoch 50, Loss: 0.0885
Epoch 60, Loss: 0.0761
Epoch 70, Loss: 0.0670
Epoch 80, Loss: 0.0600
Epoch 90, Loss: 0.0544
Epoch 100, Loss: 0.0499
Probabilities: [0.98005011 0.01994989]
Predicted class: 0
